In [ ]:
from __future__ import annotations

import io
import os
import re
from pathlib import Path

import pandas as pd

METRIC_NAMES = ["Mean", "Median", "P90"]
RUN_DIR_PATTERN = re.compile(r"^(?P<parameter_set>.+)_res_\d+_k_views_\d+$")

def find_workspace_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "train_parallel.py").exists() and (candidate / "functions").exists():
            return candidate
    raise FileNotFoundError("Could not locate the GeoPAS workspace root from the current working directory.")

WORKSPACE_ROOT = find_workspace_root(Path.cwd().resolve())
PROJECT_ROOT = Path(
    os.environ.get("PROJECT_ROOT", os.environ.get("GEOPAS_PROJECT_ROOT", str(WORKSPACE_ROOT.parent)))
).resolve()
DEFAULT_RESULT_ROOT = PROJECT_ROOT / "results" / "bbob_by_deepela" / "results"

RESULT_ROOT = Path(os.environ.get("RESULTS_ROOT", str(DEFAULT_RESULT_ROOT))).resolve()
GRID_ROOT = RESULT_ROOT / "bbob"
SPLITS = ["LPO", "LIO", "RANDOM"]
FILENAME_GLOB = "res_priorscale_*.csv"
SHORTEN_SECTION_NAMES = False

def _extract_table_block(lines: list[str], stat: str, substat: str) -> pd.DataFrame:
    start = None
    for index in range(len(lines) - 1):
        if lines[index].strip() == stat and lines[index + 1].strip() == substat:
            start = index + 2
            break
    if start is None:
        raise ValueError(f"Could not find section '{stat} -> {substat}'")

    header = lines[start].strip()
    if not header:
        raise ValueError(f"Empty header for section '{stat} -> {substat}'")

    data_lines: list[str] = []
    for index in range(start + 1, len(lines)):
        if lines[index].strip() == "":
            break
        data_lines.append(lines[index])

    block = "\n".join([header] + data_lines)
    table = pd.read_csv(io.StringIO(block))
    if table.shape[1] < 2:
        raise ValueError(f"Parsed table for '{stat} -> {substat}' looks wrong: columns={table.columns.tolist()}")

    return table.set_index(table.columns[0])

def read_as_tables(csv_path: Path) -> dict[str, pd.DataFrame]:
    lines = csv_path.read_text(encoding="utf-8").splitlines()
    return {metric: _extract_table_block(lines, metric, "AS") for metric in METRIC_NAMES}

def _maybe_shorten(section: str) -> str:
    if not SHORTEN_SECTION_NAMES:
        return section
    return section.replace("_priorscalesigmoid_log", "").replace("_tailscale1.0", "")

def _extract_parameter_set(path_token: str) -> str:
    match = RUN_DIR_PATTERN.fullmatch(path_token)
    parameter_set = match.group("parameter_set") if match else path_token
    return _maybe_shorten(parameter_set)

def _iter_split_csv_paths(split: str) -> list[Path]:
    split_root = GRID_ROOT / split.lower()
    if not split_root.is_dir():
        raise FileNotFoundError(f"Split directory does not exist: {split_root}")

    csv_paths = sorted(split_root.glob(f"*/{FILENAME_GLOB}"))
    if not csv_paths:
        raise FileNotFoundError(
            f"No aggregated-over-seeds CSVs found for split={split} under {split_root} via */{FILENAME_GLOB}"
        )
    return csv_paths

def _build_section_name(csv_path: Path, split: str) -> str:
    parts = csv_path.relative_to(GRID_ROOT).parts
    split_lower = split.lower()
    if len(parts) != 3 or parts[0].lower() != split_lower:
        raise ValueError(
            "Expected aggregated path layout <split>/<run_dir>/<csv>; "
            f"got {parts} for {csv_path}"
        )

    run_dir_name = parts[1]
    return " -- ".join([_extract_parameter_set(run_dir_name), csv_path.stem])

def build_sectioned_csv_for_split(split: str) -> Path:
    csv_paths = _iter_split_csv_paths(split)
    out_path = RESULT_ROOT / f"AS_mean_median_p90__{split}__ALL_RUNS.csv"
    bad: list[tuple[Path, str]] = []
    wrote = 0

    with out_path.open("w", encoding="utf-8", newline="\n") as handle:
        for csv_path in csv_paths:
            section = _build_section_name(csv_path, split)
            try:
                tables = read_as_tables(csv_path)
            except Exception as exc:
                bad.append((csv_path, str(exc)))
                continue

            handle.write(f"{section}\n")
            for metric in METRIC_NAMES:
                handle.write(f"{metric}\n")
                tables[metric].to_csv(handle)
            handle.write("\n")
            wrote += 1

    if bad:
        print(f"\n[WARN] Failed to parse {len(bad)} file(s) for split={split}:")
        for path, message in bad[:10]:
            print(f"  - {path}: {message}")
        if len(bad) > 10:
            print(f"  ... and {len(bad) - 10} more")

    print(f"Wrote {out_path} with {wrote} aggregated run(s).")
    return out_path

def build_all_available_splits(splits: list[str]) -> dict[str, Path]:
    outputs: dict[str, Path] = {}
    missing: list[str] = []

    for split in splits:
        try:
            outputs[split] = build_sectioned_csv_for_split(split)
        except FileNotFoundError as exc:
            print(f"[WARN] {exc}")
            missing.append(split)

    if not outputs:
        raise FileNotFoundError(f"No aggregated result CSVs found under {GRID_ROOT} for any of {splits}")

    if missing:
        print(f"[WARN] Skipped unavailable splits: {', '.join(missing)}")

    return outputs

outs = build_all_available_splits(SPLITS)
outs

Wrote /data1/home/jw1017/AS_BBO_REBUILT/results/bbob_by_deepela/results/AS_mean_median_p90__LPO__ALL_RUNS.csv with 121 aggregated run(s).
Wrote /data1/home/jw1017/AS_BBO_REBUILT/results/bbob_by_deepela/results/AS_mean_median_p90__LIO__ALL_RUNS.csv with 121 aggregated run(s).
Wrote /data1/home/jw1017/AS_BBO_REBUILT/results/bbob_by_deepela/results/AS_mean_median_p90__RANDOM__ALL_RUNS.csv with 121 aggregated run(s).


{'LPO': PosixPath('/data1/home/jw1017/AS_BBO_REBUILT/results/bbob_by_deepela/results/AS_mean_median_p90__LPO__ALL_RUNS.csv'),
 'LIO': PosixPath('/data1/home/jw1017/AS_BBO_REBUILT/results/bbob_by_deepela/results/AS_mean_median_p90__LIO__ALL_RUNS.csv'),
 'RANDOM': PosixPath('/data1/home/jw1017/AS_BBO_REBUILT/results/bbob_by_deepela/results/AS_mean_median_p90__RANDOM__ALL_RUNS.csv')}

In [2]:
from __future__ import annotations

import os
from pathlib import Path


try:
    RESULT_ROOT
except NameError:
    def find_workspace_root(start: Path) -> Path:
        for candidate in (start, *start.parents):
            if (candidate / "train_parallel.py").exists() and (candidate / "functions").exists():
                return candidate
        raise FileNotFoundError("Could not locate the GeoPAS workspace root from the current working directory.")

    WORKSPACE_ROOT = find_workspace_root(Path.cwd().resolve())
    PROJECT_ROOT = Path(
        os.environ.get("PROJECT_ROOT", os.environ.get("GEOPAS_PROJECT_ROOT", str(WORKSPACE_ROOT.parent)))
    ).resolve()
    DEFAULT_RESULT_ROOT = PROJECT_ROOT / "results" / "bbob_by_deepela" / "results"
    RESULT_ROOT = Path(os.environ.get("RESULTS_ROOT", str(DEFAULT_RESULT_ROOT))).resolve()


PRIMARY_METRICS = ("Mean", "Median", "P90")
OPTIONAL_METRICS = ("Log_Mean", "Log_Median", "Log_P90")
ALL_METRICS = PRIMARY_METRICS + OPTIONAL_METRICS
ALL_METRIC_SET = set(ALL_METRICS)
SPLIT_NAMES = list(SPLITS) if "SPLITS" in globals() else ["LPO", "LIO", "RANDOM"]
FILES = {split: RESULT_ROOT / f"AS_mean_median_p90__{split}__ALL_RUNS.csv" for split in SPLIT_NAMES}
OUT = RESULT_ROOT / "AS_mean_median_p90__MERGED__ALL_RUNS.csv"


def _consume_metric_block(lines: list[str], index: int, metric: str, path: Path) -> tuple[list[str], int]:
    if index >= len(lines) or lines[index].strip() != metric:
        raise ValueError(f"{path}: expected '{metric}' at line {index + 1}")

    index += 1
    rows: list[str] = []
    while index < len(lines):
        token = lines[index].strip()
        if token == "" or token in ALL_METRIC_SET:
            break
        rows.append(lines[index])
        index += 1
    return rows, index


def parse_sectioned_csv(path: Path) -> dict[str, dict[str, list[str]]]:
    lines = path.read_text(encoding="utf-8").splitlines()
    out: dict[str, dict[str, list[str]]] = {}
    index = 0
    total = len(lines)

    def skip_blanks(cursor: int) -> int:
        while cursor < total and lines[cursor].strip() == "":
            cursor += 1
        return cursor

    while True:
        index = skip_blanks(index)
        if index >= total:
            break

        section = lines[index].strip()
        index += 1
        metrics: dict[str, list[str]] = {}

        for metric in PRIMARY_METRICS:
            index = skip_blanks(index)
            metrics[metric], index = _consume_metric_block(lines, index, metric, path)

        for metric in OPTIONAL_METRICS:
            index = skip_blanks(index)
            if index < total and lines[index].strip() == metric:
                metrics[metric], index = _consume_metric_block(lines, index, metric, path)
            else:
                metrics[metric] = []

        out[section] = metrics

    return out


def get_last_metric_value(rows: list[str]) -> float:
    if not rows:
        return float("inf")
    return float(rows[-1].rsplit(",", 1)[-1])


available_files = {split: path for split, path in FILES.items() if path.exists()}
missing_files = [split for split, path in FILES.items() if not path.exists()]

if not available_files:
    raise FileNotFoundError(
        "No per-split CSVs found. Run Cell 1 first, or update FILES to point to existing outputs."
    )

if missing_files:
    print(f"[WARN] Missing split outputs: {', '.join(missing_files)}")

parsed = {split: parse_sectioned_csv(path) for split, path in available_files.items()}
order = list(parsed[next(iter(parsed))].keys())

if "LPO" in parsed:
    original_index = {section: index for index, section in enumerate(order)}
    order.sort(
        key=lambda section: (
            section not in parsed["LPO"],
            get_last_metric_value(parsed["LPO"][section]["Median"]) if section in parsed["LPO"] else float("inf"),
            original_index[section],
        )
    )

missing_section_counts = {split: 0 for split in available_files}
for section in order:
    for split in available_files:
        if section not in parsed[split]:
            missing_section_counts[split] += 1

reported_missing = {split: count for split, count in missing_section_counts.items() if count > 0}
if reported_missing:
    print("[WARN] Missing sections by split:")
    for split, count in reported_missing.items():
        print(f"  - {split}: {count}")

with OUT.open("w", encoding="utf-8", newline="\n") as handle:
    for section in order:
        handle.write(section + "\n")
        for split in available_files:
            if section not in parsed[split]:
                continue

            handle.write(split + "\n")
            for metric in ALL_METRICS:
                rows = parsed[split][section].get(metric, [])
                if not rows:
                    continue
                handle.write(metric + "\n")
                handle.write("\n".join(rows) + "\n")
        handle.write("\n")

print(f"Wrote {OUT}")

Wrote /data1/home/jw1017/AS_BBO_REBUILT/results/bbob_by_deepela/results/AS_mean_median_p90__MERGED__ALL_RUNS.csv
